# Extract data from CERCA raw files

In [4]:
from docx import Document
import pandas as pd
from google.cloud import bigquery
import requests
from tqdm import tqdm
import time
from df2gspread import gspread2df as g2d

In [ ]:
interest_centers = ['CRM', 'ICRA', 'CTFC']
file_path = '../data/external/5_Bibliometria_SIRIS_xx1025/'

## Extract DOI list

**Each file has a different format so we go center by center**

### CRM

In [ ]:
center_name = 'CRM/'
file_name = 'XX'

In [ ]:
### ÉS UN PDF
df_crm

### ICRA

In [ ]:
center_name = 'ICRA/'
file_name = 'XXX'

In [ ]:
### ÉS UN PDF
df_icra

### CTFC

In [5]:
center_name = 'CTFC/'
file_name = 'XXX'

In [ ]:
### PÈNDENTS DE LES DADES
df_ctfc

,DOI,Center
0,10.59277/ROMJIST.2024.2.09,ICN2
1,10.1002/ece2.12,ICN2
2,10.1093/mam/ozae044.520,ICN2
3,10.1103/physrevlett.132.266301,ICN2
4,10.7203/metode.15.27225,ICN2
...,...,...
823,10.1103/PhysRevB.108.054524,ICN2
824,10.1021/acsaem.1c02919,ICN2
825,10.1016/j.synthmet.2021.116844,ICN2
826,10.1038/s42254-021-00318-1,ICN2


In [ ]:
df_centers = pd.concat([df_crm, df_icra, df_ctfc], ignore_index = True)
df_centers.to_csv('../data/processed/CERCA_5_Bibliometria_SIRIS_segonaentrega.csv', index = False)
df_centers

,DOI,Center
0,10.1016/j.scitotenv.2023.168824,BETA
1,10.23818/limn.43.07,BETA
2,10.1016/j.aquatox.2024.106843,BETA
3,10.3390/agronomy14050935,BETA
4,10.32347/2077-3455.2024.68.215-227,BETA
...,...,...
10081,10.7759/cureus.13183,ResearchMar
10082,10.7759/cureus.16472,ResearchMar
10083,10.7759/cureus.40708,ResearchMar
10084,10.7759/cureus.62509,ResearchMar


In [21]:
df_centers.DOI.nunique()

8820

## Check which publications are not in OA using DOI

In [27]:
PROJECT_ID = 'siris-datasets'
DATASET_ID = 'openalex'

def bg_query(query):
    client = bigquery.Client(project=PROJECT_ID)
    df = client.query(query)
    return df.to_dataframe()

In [28]:
in_query = str(tuple(df_centers.DOI.tolist()))
in_query = in_query.replace(',)', ')')

sql = f"""SELECT ww.DOI,
       a.display_name,
       wa.author_order,
       wa.author_position,
       wa.is_corresponding,
       wins.ID AS institution_id,
       wins.COUNTRY_CODE
      FROM `{PROJECT_ID}.{DATASET_ID}.works` ww
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships` wa ON wa.WORK_ID = ww.ID
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.institutions` wins ON wins.ID = wa.INSTITUTION_ID
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.authors` a ON a.ID = wa.author_id
      WHERE ww.DOI IN {in_query}
      """

df_OA = bg_query(sql).dropna(subset = 'DOI').reset_index(drop = True)
df_OA

/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE
0,10.1038/s41591-023-02610-2,José R. Banegas,88,middle,False,4210154894,ES
1,10.1038/s41591-023-02610-2,Charles Agyemang,47,middle,False,887064364,NL
2,10.1038/s41591-023-02610-2,Andrew Wong,750,middle,False,45129253,GB
3,10.1038/s41591-023-02610-2,Farhad Zamani,766,middle,False,161106909,IR
4,10.1038/s41591-023-02610-2,José R. Banegas,88,middle,False,63634437,ES
...,...,...,...,...,...,...,...
189137,10.1111/jvh.13412,Karine Lacombe,33,middle,False,4210133907,FR
189138,10.1111/jvh.13412,Karine Lacombe,33,middle,False,4210097159,FR
189139,10.1111/jvh.13412,Karine Lacombe,33,middle,False,39804081,FR
189140,10.1002/lol2.10277,Maria Lundgren,33,middle,False,223464139,SE


In [29]:
(df_OA.DOI.nunique()) / df_centers.DOI.nunique()

0.9305475504322767

### Identify CERCA authors using OA (93% of the dataset)

- By affiliation ID
- By raw affiliations
  - Using parents affiliations
  - Manually checking above > 1 per raw affiliation
  - String search with keywords for = 1 doi per raw affilation

In [ ]:
cerca_centers = {'CRM' : ['4210122226'], # NOT IN OA
                    'ICRA' : ['2799562678'],
                    'CTFC' : ['4210117018']}

interest_cerca = list(set(sum(list(cerca_centers.values()), [])))
interest_cerca = [int(x) for x in interest_cerca if x != '']

cerca_authors = df_OA[df_OA.institution_id.isin(interest_cerca)].drop_duplicates(['DOI'])

cerca_parents = {'CRM' : ['123044942'],
                    'ICRA' : ['251424209'],
                    'CTFC' : ['15766328', '123044942']}

parents_cerca = list(set(sum(list(cerca_parents.values()), [])))
parents_cerca = [int(x) for x in parents_cerca if x != '']

cerca_possible_authors = df_OA[(df_OA.institution_id.isin(parents_cerca)) & (~df_OA.display_name.isin(cerca_authors.display_name))].drop_duplicates(['DOI'])
cerca_possible_authors

,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE
162,10.1016/j.ijid.2023.10.019,José Marı́a Tolosana,142,middle,False,71999127,ES
321,10.1016/j.jpsychires.2022.02.009,Silvia Sola,140,middle,False,123044942,ES
464,10.1007/s10120-024-01504-7,Leticia Moreira,35,last,True,71999127,ES
588,10.1016/j.cgh.2020.11.002,Leticia Moreira,50,middle,False,71999127,ES
755,10.1016/j.jmoldx.2022.02.003,Mar Costa,84,middle,False,123044942,ES
...,...,...,...,...,...,...,...
184091,10.1016/j.jaip.2024.03.050,Sami Aqel,29,middle,False,123044942,ES
184486,10.1136/ard-2023-224990,Núria Guañabens,29,middle,False,71999127,ES
184636,10.1111/apt.18230,Xavier Calvet,29,middle,False,123044942,ES
186635,10.2139/ssrn.4139263,Albert Palou Vilar,31,middle,False,115304662,ES


In [31]:
in_query = str(tuple(cerca_possible_authors.DOI.tolist()))
in_query = in_query.replace(',)', ')')

sql = f"""SELECT ww.DOI,
       a.display_name,
       war.raw_affiliation,
       wins.ID AS institution_id,
       wins.COUNTRY_CODE
      FROM `{PROJECT_ID}.{DATASET_ID}.works` ww
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships` wa ON wa.WORK_ID = ww.ID
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.institutions` wins ON wins.ID = wa.INSTITUTION_ID
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.authors` a ON a.ID = wa.author_id
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships_raw` war ON war.WORK_ID = ww.ID AND war.AUTHOR_ID = wa.AUTHOR_ID
      WHERE ww.DOI IN {in_query}
      """
df_possible_cerca = bg_query(sql).dropna(subset = 'DOI').drop_duplicates(['DOI', 'raw_affiliation']).reset_index(drop = True).reset_index(drop = True)
df_possible_cerca

/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,DOI,display_name,raw_affiliation,institution_id,COUNTRY_CODE
0,10.1007/s13304-024-01859-7,M Serrano-Navidad,None,<NA>,None
1,10.1007/s00192-022-05365-5,Carolina Vidal Gamboa,Public Health Institute of the Andrés Bello Un...,13897259,CL
2,10.1111/jcmm.16736,Lejla Pojskić,Institute for Genetic Engineering and Biotechn...,104817121,BA
3,10.1038/s41420-024-01857-z,Marina Mola-Caminal,"Neurology, Hospital del Mar Medical Research I...",4210130874,ES
4,10.1371/journal.pone.0265079,José Manuel Santos‐Lozano,Centro de Investigación Biomédica en Red Fisio...,4210166912,ES
...,...,...,...,...,...
32437,10.1016/j.gaceta.2021.01.005,Jorge Luis Díaz,Unidad Docente de Medicina Preventiva y Salud ...,170486558,ES
32438,10.1016/j.biopha.2024.116857,Anne Hansen Ree,"Oncobell Program, Bellvitge Institute for Biom...",2800192669,NO
32439,10.1016/j.biopha.2024.116857,Anne Hansen Ree,"ProCURE Program, Catalan Institute of Oncology...",2800192669,NO
32440,10.1016/j.biopha.2024.116857,Anne Hansen Ree,"University of Oslo, Problemveien 11, Oslo 0313...",2800192669,NO


In [ ]:
grouped = df_possible_cerca[df_possible_cerca.COUNTRY_CODE == 'ES'].groupby('raw_affiliation').count().sort_values('DOI', ascending = False)[['DOI']]
to_check = grouped[grouped.DOI > 1]
print(df_possible_cerca[df_possible_cerca.raw_affiliation.isin(to_check.index)].DOI.nunique())
# to_check.to_csv('to_check_v2.csv')
to_check
### WE MISS 876 DOIS BY FILTERING CHECKING THE THRESHOLD OF 1 AND MANUALLY CHECK THE MISSING

1542


,DOI
raw_affiliation,
"Universitat Pompeu Fabra (UPF), Barcelona, Spain",68
"IMIM (Hospital del Mar Medical Research Institute), Barcelona, Spain",57
"ISGlobal, Barcelona, Spain",49
"Universitat Pompeu Fabra, Barcelona, Spain",41
"Department of Preventive Medicine, University of Valencia, Valencia, Spain",40
...,...
"Universidad Francisco de Vitoria, Madrid, Spain",2
"Bellvitge University Hospital, Department of Psychiatry, Bellvitge Biomedical Research Institute (IDIBELL), Neurosciences Group - Psychiatry and Mental Health, Barcelona, Spain",2
"Universitat Rovira i Virgili, Departament de Bioquímica i Biotecnologia, Alimentació, Nutrició, Desenvolupament i Salut Mental (ANUT-DSM), Unitat de Nutrició Humana, Reus, Spain",2


In [37]:
df_check = g2d.download('1DtbOJzE9c9xCtuMfQQX8f4Uom6U7Mx7wwmzri57Xj4A', '>1', col_names = True, row_names = False)

compute  = df_possible_cerca[df_possible_cerca.raw_affiliation.isin(df_check[df_check.CERCA == 'TRUE'].raw_affiliation)]

len(set(list(compute.DOI.unique()) + list(cerca_authors.DOI.unique()))) / df_centers.DOI.nunique()

Not all requested scopes were granted by the authorization server, missing scopes https://spreadsheets.google.com/feeds, https://docs.google.com/feeds.


0.5674351585014409

In [ ]:
cerca_string = {'BETA' : ['vic', 'beta'], # NOT IN OA
                    'CREAF' : ['creaf', 'cerdanyola', 'bellaterra'],
                    'ICN2' : ['icn2', 'bellaterra', 'cerdanyola'],
                    'ISGlobal' : ['global'],
                    'ResearchMar' : ['mar', 'imim']}
cerca_string = list(set(sum(list(cerca_string.values()), [])))


df_tmp = grouped[(grouped.DOI == 1) & (~grouped.index.isin(df_check.raw_affiliation))].reset_index()
df_tmp['raw_affiliation'] = df_tmp['raw_affiliation'].str.lower()

df_tmp_1 = df_tmp[df_tmp['raw_affiliation'].str.contains(('|'.join(cerca_string)), case=False, na=False)].drop_duplicates().reset_index(drop = True)
# df_tmp_1.to_csv('to_check_v2_2.csv')
df_tmp_1

,raw_affiliation,DOI
0,"internal medicine department, imim (hospital d...",1
1,integrative pharmacology and systems neuroscie...,1
2,integrative pharmacology and systems neuroscie...,1
3,integrated pharmacology and systems neuroscien...,1
4,integrative pharmacology and systems neuroscie...,1
...,...,...
4541,"department of neuroradiology, hospital del mar...",1
4542,"department of neuroradiology, hospital del mar...",1
4543,"department of neuropsychiatry and addictions, ...",1
4544,"department of neuropsychiatry and addictions, ...",1


In [39]:
df_check_v2 = g2d.download('1DtbOJzE9c9xCtuMfQQX8f4Uom6U7Mx7wwmzri57Xj4A', '=1', col_names = True, row_names = False)
df_check_v2

Not all requested scopes were granted by the authorization server, missing scopes https://spreadsheets.google.com/feeds, https://docs.google.com/feeds.


,raw_affiliation,DOI,CERCA,CERCA_IMIM
0,". clínica de rehabilitación de salud mental, d...",1,FALSE,
1,. department of child and adolescent psychiatr...,1,FALSE,FALSE
2,". department of psychiatry, chinese university...",1,FALSE,FALSE
3,. hospital del mar medical research institute ...,1,TRUE,
4,". infectious diseases service, hospital del ma...",1,TRUE,
...,...,...,...,...
4543,wildlife ecology & health group (we&h) and ser...,1,FALSE,
4544,"wildlife ecology & health group (we&h), and se...",1,FALSE,
4545,"wildlife ecology & health group (we&h), servei...",1,FALSE,
4546,wildlife ecology & health research group (we&h...,1,FALSE,


In [40]:
check_1 = df_possible_cerca[df_possible_cerca.raw_affiliation.isin(df_check[df_check.CERCA == 'TRUE'].raw_affiliation)]
check_1_imim = df_possible_cerca[df_possible_cerca.raw_affiliation.isin(df_check[df_check.CERCA_IMIM == 'TRUE'].raw_affiliation)]
check_2 = df_possible_cerca[df_possible_cerca.raw_affiliation.str.lower().isin(df_check_v2[df_check_v2.CERCA == 'TRUE'].raw_affiliation)]
check_2_imim = df_possible_cerca[df_possible_cerca.raw_affiliation.str.lower().isin(df_check_v2[df_check_v2.CERCA_IMIM == 'TRUE'].raw_affiliation)]

cerca_doi = list(set(
    list(cerca_authors.DOI.unique()) +
    # list(cerca_possible_authors.DOI.unique()) +
    list(check_1.DOI.unique()) +
    list(check_2.DOI.unique()) +
    list(check_1_imim.DOI.unique()) +
    list(check_2_imim.DOI.unique())
))
len(cerca_doi) / df_centers.DOI.nunique()

0.6812680115273775

In [41]:
cerca_possible_authors_not_parent = df_ResearchMar[~df_ResearchMar.DOI.isin(cerca_doi)]

in_query = str(tuple(cerca_possible_authors_not_parent.DOI.tolist()))
in_query = in_query.replace(',)', ')')

sql = f"""SELECT ww.DOI,
       a.display_name,
       war.raw_affiliation,
       wins.ID AS institution_id,
       wins.COUNTRY_CODE
      FROM `{PROJECT_ID}.{DATASET_ID}.works` ww
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships` wa ON wa.WORK_ID = ww.ID
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.institutions` wins ON wins.ID = wa.INSTITUTION_ID
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.authors` a ON a.ID = wa.author_id
      LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.works_authorships_raw` war ON war.WORK_ID = ww.ID AND war.AUTHOR_ID = wa.AUTHOR_ID
      WHERE ww.DOI IN {in_query}
      """
df_possible_cerca_not_parent = bg_query(sql).dropna(subset = 'DOI').drop_duplicates(['DOI', 'raw_affiliation']).reset_index(drop = True).reset_index(drop = True)
df_possible_cerca_not_parent

/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/home/siris/2025CERCA01/2025CERCA01_env/lib/python3.10/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,DOI,display_name,raw_affiliation,institution_id,COUNTRY_CODE
0,10.1177/19476035211053827,Demirhan Dıraçoğlu,Department of Physical Medicine and Rehabilita...,67581229,TR
1,10.1016/j.thromres.2021.06.008,Takao Kato,"Department of Cardiovascular Medicine, Graduat...",22299242,JP
2,10.1002/hem3.70004,Alicia Enrico,Hospital Italiano La Plata Buenos Aires Argentina,4210118619,AR
3,10.1002/hem3.70004,Deepesh Lad,Department of Internal Medicine Postgraduate I...,45294948,IN
4,10.1111/jdv.18574,Ece Nur Değirmentepe,"Okmeydani Training and Research Hospital, Ista...",4210155168,TR
...,...,...,...,...,...
17614,10.1093/hmg/ddac084,Stephen R. Braddock,"Division of Medical Genetics, Department of Pe...",47838141,US
17615,10.1002/alz.12662,Nancy S. Foldi,"Department of Psychiatry, New York University ...",40889946,US
17616,10.1002/alz.12662,Nancy S. Foldi,"Department of Psychology, Queens College and T...",40889946,US
17617,10.1016/j.esmoop.2024.103972,Jean Hoffman‐Censits,Johns Hopkins Sidney Kimmel Comprehensive Canc...,4210164401,US


In [42]:
grouped = df_possible_cerca_not_parent[df_possible_cerca_not_parent.COUNTRY_CODE == 'ES'].groupby('raw_affiliation').count().sort_values('DOI', ascending = False)[['DOI']]
to_check = grouped[grouped.DOI > 0]
to_check.to_csv( 'to_check_v3.csv' )
to_check

,DOI
raw_affiliation,
"Hospital del Mar, Barcelona, Spain",41
"Hospital Universitario 12 de Octubre, Madrid, Spain",13
"Department of Dermatology, Hospital del Mar, Barcelona, Spain",13
"Hospital Universitario Central de Asturias, Oviedo, Spain",12
"Medical Oncology Department, Hospital del Mar, Barcelona, Spain",11
...,...
"Department of Surgery, Hospital Universitario Miguel Servet, 50009 Zaragoza, Spain",1
"Department of Surgery, Hospital Universitario Marqués de Valdecilla, 39008 Santander, Spain",1
"Department of Surgery, Hospital Universitario Lozano Blesa, 50009 Zaragoza, Spain",1


In [43]:
df_check_v3 = g2d.download('1DtbOJzE9c9xCtuMfQQX8f4Uom6U7Mx7wwmzri57Xj4A', 'Nonparent', col_names = True, row_names = False)
df_check_v3

Not all requested scopes were granted by the authorization server, missing scopes https://spreadsheets.google.com/feeds, https://docs.google.com/feeds.


,raw_affiliation,DOI,CERCA
0,"Hospital del Mar, Barcelona, Spain",41,TRUE
1,"Department of Dermatology, Hospital del Mar, B...",13,TRUE
2,"Hospital Universitario 12 de Octubre, Madrid, ...",13,FALSE
3,"Hospital Universitario Central de Asturias, Ov...",12,FALSE
4,"Medical Oncology Department, Hospital del Mar,...",11,TRUE
...,...,...,...
8398,"Department of Surgery, Hospital Universitario ...",1,FALSE
8399,"Department of Surgery, Hospital Universitario ...",1,FALSE
8400,"Department of Surgery, Hospital Universitario ...",1,FALSE
8401,"Department of Surgery, Hospital Universitario ...",1,FALSE


In [44]:
check_1 = df_possible_cerca[df_possible_cerca.raw_affiliation.isin(df_check[df_check.CERCA == 'TRUE'].raw_affiliation)]
check_1_imim = df_possible_cerca[df_possible_cerca.raw_affiliation.isin(df_check[df_check.CERCA_IMIM == 'TRUE'].raw_affiliation)]
check_2 = df_possible_cerca[df_possible_cerca.raw_affiliation.str.lower().isin(df_check_v2[df_check_v2.CERCA == 'TRUE'].raw_affiliation)]
check_2_imim = df_possible_cerca[df_possible_cerca.raw_affiliation.str.lower().isin(df_check_v2[df_check_v2.CERCA_IMIM == 'TRUE'].raw_affiliation)]
check_3 = df_possible_cerca_not_parent[df_possible_cerca_not_parent.raw_affiliation.isin(df_check_v3[df_check_v3.CERCA == 'TRUE'].raw_affiliation)]


cerca_doi = list(set(
    list(cerca_authors.DOI.unique()) +
    # list(cerca_possible_authors.DOI.unique()) +
    list(check_1.DOI.unique()) +
    list(check_2.DOI.unique()) +
    list(check_1_imim.DOI.unique()) +
    list(check_2_imim.DOI.unique()) + 
    list(check_3.DOI.unique())
))
len(cerca_doi) / df_centers.DOI.nunique()

0.8435158501440922

In [45]:
df_check = pd.concat((cerca_authors, check_1, check_2, check_1_imim, check_2_imim, check_3))
df_check

,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE,raw_affiliation
282,10.1088/2515-7639/acc893,Masoud Karimipour,35,middle,False,4210093216,ES,NaN
593,10.1016/j.jval.2021.03.021,Luis Gları́a,47,middle,False,4210156109,ES,NaN
739,10.1016/j.euroneuro.2024.05.006,Alba Toll,35,middle,False,4210156109,ES,NaN
915,10.1007/s10654-021-00733-9,Martine Vrijheid,35,middle,False,4210148332,ES,NaN
934,10.1111/acel.14194,Martine Vrijheid,52,middle,False,4210148332,ES,NaN
...,...,...,...,...,...,...,...,...
14231,10.1093/bjd/ljae067,Ana M. Giménez‐Arnau,<NA>,NaN,<NA>,4210130874,ES,"Hospital Del Mar Research Institute, Universit..."
14238,10.1016/j.transproceed.2021.09.013,Marta Crespo,<NA>,NaN,<NA>,4210130874,ES,"Department of Nephrology, Hospital del Mar, Ba..."
14241,10.1016/j.avsg.2021.10.054,Albert Benet i Clarà,<NA>,NaN,<NA>,4210115082,ES,Department of Vascular and Endovascular Surger...
14246,10.1016/j.chest.2023.04.029,Esther Barreiro,<NA>,NaN,<NA>,2801357902,ES,"Servicio de Neumología, Hospital del Mar-IMIM,..."


In [46]:
# HI HA ERROR AMB L'ASSIGNACIÓ I PER AIXÒ SURT RAR! S'HA DE FER ALS POSSIBLE (PATENT I NO PARENT) I LLAVORS FER EL MERGE AMB EL OA QUE TÉ L'AUTHOR ORDER

df_OA['CERCA'] = (df_OA['display_name'].isin(df_check['display_name']) & df_OA['DOI'].isin(df_check['DOI']))
df_final = df_OA.merge(df_centers, on = 'DOI').drop_duplicates().reset_index(drop = True)
df_final.to_csv('../data/processed/CERCA_5_Bibliometria_SIRIS_OA.csv', index = False)
df_final

,DOI,display_name,author_order,author_position,is_corresponding,institution_id,COUNTRY_CODE,CERCA,Center
0,10.1038/s41591-023-02610-2,José R. Banegas,88,middle,False,4210154894,ES,False,ResearchMar
1,10.1038/s41591-023-02610-2,Charles Agyemang,47,middle,False,887064364,NL,False,ResearchMar
2,10.1038/s41591-023-02610-2,Andrew Wong,750,middle,False,45129253,GB,False,ResearchMar
3,10.1038/s41591-023-02610-2,Farhad Zamani,766,middle,False,161106909,IR,False,ResearchMar
4,10.1038/s41591-023-02610-2,José R. Banegas,88,middle,False,63634437,ES,False,ResearchMar
...,...,...,...,...,...,...,...,...,...
193398,10.1111/jvh.13412,Karine Lacombe,33,middle,False,4210133907,FR,False,ISGlobal
193399,10.1111/jvh.13412,Karine Lacombe,33,middle,False,4210097159,FR,False,ISGlobal
193400,10.1111/jvh.13412,Karine Lacombe,33,middle,False,39804081,FR,False,ISGlobal
193401,10.1002/lol2.10277,Maria Lundgren,33,middle,False,223464139,SE,False,BETA


In [47]:
df_final[df_final.CERCA == True].drop_duplicates(['DOI', 'Center']).groupby('Center').size()

Center
BETA             50
CREAF           825
ICN2            712
ISGlobal        626
ResearchMar    3725
dtype: int64